In [ ]:
try:
  import google.colab
  IN_COLAB = True
except:
  IN_COLAB = False

In [ ]:
if IN_COLAB:
  # Install dependencies
  ! pip install --upgrade pip
  ! pip install czitools[ndv]
  ! pip install ipyfilechooser

In [ ]:
# import the required libraries
from czitools.metadata_tools import czi_metadata as czimd
from czitools.read_tools import read_tools as czird
from czitools.utils import misc
from czitools.utils.napari_tools import display_xarray_in_napari
from czitools.utils.ndv_tools import create_luts_ndv, create_scales_ndv
from pathlib import Path
from ipyfilechooser import FileChooser
from IPython.display import display, HTML
import os
import dask.array as da
import requests
import glob
import ipywidgets as widgets
import ndv
import xarray as xr
from typing import Any, Mapping, cast
from ndv._types import AxisKey


In [ ]:
# try to find the folder with data and download otherwise from GitHub.

# Folder containing the input data
if IN_COLAB:
    INPUT_FOLDER = 'data/'
if not IN_COLAB:
    INPUT_FOLDER = '../../data/'

# Path to the data on GitHub
GITHUB_IMAGES_PATH = "https://raw.githubusercontent.com/sebi06/czitools/main/data.zip"

# Download data
if not (os.path.isdir(INPUT_FOLDER)):
    compressed_data = './data.zip'
    if not os.path.isfile(compressed_data):
        import io
        response = requests.get(GITHUB_IMAGES_PATH, stream=True)
        compressed_data = io.BytesIO(response.content)

    import zipfile
    with zipfile.ZipFile(compressed_data, 'r') as zip_accessor:
        zip_accessor.extractall('./')

In [ ]:
if not IN_COLAB:
    # choose local file
    fc = FileChooser()
    fc.default_path = INPUT_FOLDER
    fc.filter_pattern = '*.czi'
    display(fc)

elif IN_COLAB:
    # list files inside the folder on gdrive
    czifiles = glob.glob(os.path.join(INPUT_FOLDER, "*.czi"))
    wd = widgets.Select(
        options=czifiles,
        description='CZI Files:',
        layout={'width': 'max-content'}
    )
    display(wd)

In [ ]:
if not IN_COLAB:
    filepath = fc.selected
elif IN_COLAB:
    filepath = wd.value

print(f"Selected File: {filepath}")

In [ ]:
# get the complete metadata at once as one big class
mdata = czimd.CziMetadata(filepath)

# get the CZI metadata dictionary directly from filename
mdict = czimd.create_md_dict_red(mdata, sort=False, remove_none=True)

# convert metadata dictionary to a pandas dataframe
mdframe = misc.md2dataframe(mdict)

# create a ipywdiget to show the dataframe with the metadata
wd = widgets.Output(layout={"scrollY": "auto", "height": "300px"})

with wd:
    display(HTML(mdframe.to_html()))
display(widgets.VBox(children=[wd]))

In [ ]:
# return array with dimension order STCZYX(A)
array6d, mdata= czird.read_6darray(filepath)

# print the shape of the array etc.
print(f"Shape: {array6d.shape}")
print(f"Dimensions: {array6d.dims}")

for k, v in array6d.attrs.items():
    print(f"{k} :  {v}")

# show dask array structure
if isinstance(array6d, da.Array):
    print(array6d)
else:
    print("Shape:", array6d.shape, "dtype:", array6d.dtype)


In [ ]:
if not IN_COLAB:

    # create NDV luts and scales from metadata
    luts = create_luts_ndv(mdata)
    scales = cast(Mapping[AxisKey, float], create_scales_ndv(mdata))

    # NDV imshow kwargs: use composite mode, channel axis is "C", and provide the luts
    imshow_kwargs = {
        "channel_mode": "composite",
        "channel_axis": "C",
        "luts": luts,
    }

In [ ]:
# NOTE: NDV Jupyter backend currently expects slider coordinates as `range` objects.
# Passing an xarray DataArray directly can fail when coordinates are not ranges.

import ndv

# Use raw array backing data to avoid xarray coordinate handling in NDV Jupyter view.
viewer_data = array6d.data if hasattr(array6d, "data") else array6d

# For STCZYX(A), C is axis index 2 in the returned array order.
jupyter_imshow_kwargs = dict(imshow_kwargs)
jupyter_imshow_kwargs["channel_axis"] = 2

# NDV Jupyter colormap selector only supports known cmap names; custom LUT names from metadata
# can raise validation errors. Disable LUT injection here for notebook compatibility.
jupyter_imshow_kwargs.pop("luts", None)

# Print clear axis mapping and expected zero-based index ranges.
axis_names = list(getattr(array6d, "dims", [f"axis_{i}" for i in range(len(viewer_data.shape))]))
print("NDV axis mapping (axis_index -> dim_name, index_range):")
for axis_index, axis_name in enumerate(axis_names):
    axis_len = int(viewer_data.shape[axis_index])
    if axis_len > 1:
        print(f"  {axis_index} -> {axis_name}, 0..{axis_len - 1}")

# Display the data in NDV with scale metadata.
viewer = ndv.imshow(viewer_data, scales=scales, **jupyter_imshow_kwargs)